In [3]:
%%bash
# Clean up old runs so the notebook is safe to rerun
rm -rf IndicLID
git clone https://github.com/AI4Bharat/IndicLID.git
cd IndicLID
# Pin to the exact AI4Bharat commit
git checkout e4bfe42923c5ad581ccb64ca9feb38c0cd572ba8
pip install fasttext -q

Cloning into 'IndicLID'...
Note: switching to 'e4bfe42923c5ad581ccb64ca9feb38c0cd572ba8'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at e4bfe42 Added .gitignore


In [4]:
import sys, torch, transformers
from importlib.metadata import version

print("--- Environment Details ---")
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
# Use metadata to get fasttext version safely
print(f"FastText version: {version('fasttext')}")
print(f"Hardware: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

--- Environment Details ---
Python version: 3.13.15
PyTorch version: 2.11.0+cu128
Transformers version: 5.16.1
FastText version: 0.9.3
Hardware: Tesla T4


In [5]:
%%bash
cd IndicLID/Inference/ai4bharat
mkdir -p models

# Download safely
wget -nc -q https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-ftn.zip
wget -nc -q https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-ftr.zip
wget -nc -q https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-bert.zip

# Extract and overwrite cleanly
unzip -o -q indiclid-ftn.zip -d models/
unzip -o -q indiclid-ftr.zip -d models/
unzip -o -q indiclid-bert.zip -d models/
rm *.zip

In [7]:
import os
import sys
import torch
import pandas as pd

# Change Python's working directory so it can find the models folder!
if os.path.basename(os.getcwd()) != 'ai4bharat':
    os.chdir('IndicLID/Inference/ai4bharat')

from IndicLID import IndicLID
from transformers import BertConfig, BertForSequenceClassification

# 1. THE TOKENIZER PATCH
def patched_IndicBERT_roman_inference(self, IndicLID_BERT_inputs, output_dict, batch_size):
    if not IndicLID_BERT_inputs:
        return output_dict

    df = pd.DataFrame(IndicLID_BERT_inputs)
    dataloader = self.get_dataloaders(df.iloc[:,0], df.iloc[:,1], batch_size)

    with torch.no_grad():
        for data in dataloader:
            batch_indices, batch_inputs = data[0], data[1]
            word_embeddings = self.IndicLID_BERT_tokenizer(batch_inputs, return_tensors="pt", padding=True, truncation=True, max_length=512)
            word_embeddings = word_embeddings.to(self.device)
            token_type_ids = word_embeddings.get('token_type_ids', torch.zeros_like(word_embeddings['input_ids']))

            batch_outputs = self.IndicLID_BERT(
                word_embeddings['input_ids'],
                token_type_ids=token_type_ids,
                attention_mask=word_embeddings['attention_mask']
            )
            _, batch_predicted = torch.max(batch_outputs.logits, 1)

            for index, text_input, pred_label, logit in zip(batch_indices, batch_inputs, batch_predicted, batch_outputs.logits):
                output_dict[index] = (text_input, self.IndicLID_lang_code_dict_reverse[pred_label.item()], logit[pred_label.item()].item(), 'IndicLID-BERT')
    return output_dict

IndicLID.IndicBERT_roman_inference = patched_IndicBERT_roman_inference

# 2. THE ULTIMATE UPGRADE PATCH WITH VERIFICATION
print("Loading legacy IndicLID model...")
model = IndicLID(input_threshold=0.5, roman_lid_threshold=0.6)

print("Upgrading model architecture to modern transformers...")
state_dict = model.IndicLID_BERT.state_dict()
config_dict = model.IndicLID_BERT.config.to_dict()

new_config = BertConfig(**config_dict)
fresh_model = BertForSequenceClassification(new_config)

# Capture the mismatch keys for verification
load_result = fresh_model.load_state_dict(state_dict, strict=False)

print("\n--- Legacy Weight Transfer Verification ---")
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

allowed_unexpected = ["bert.embeddings.position_ids"]
unexplained = [k for k in load_result.unexpected_keys if k not in allowed_unexpected]

if unexplained:
    raise ValueError(f"Found unexplained unexpected keys: {unexplained}")
if load_result.missing_keys:
    raise ValueError(f"Found missing keys in the modern model: {load_result.missing_keys}")
print("Weight transfer verified successfully! Only expected legacy buffers were discarded.")

fresh_model.to(model.device)
fresh_model.eval()
model.IndicLID_BERT = fresh_model

Loading legacy IndicLID model...


config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 7.75MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Upgrading model architecture to modern transformers...

--- Legacy Weight Transfer Verification ---
Missing keys: []
Unexpected keys: ['bert.embeddings.position_ids']
Weight transfer verified successfully! Only expected legacy buffers were discarded.


In [10]:
# --- FULL INFERENCE ENTRY POINT SMOKE TESTS ---
print("\n--- Running Smoke Tests ---")

test_cases = [
    ("ഞാൻ ഇന്ന് ബിസി ആണ്", "mal_Mlym"),      # Native Malayalam
    ("Njan innu busy aanu", "mal_Latn"),        # Roman Malayalam
    ("I am busy today", "eng_Latn"),            # English
    ("phone evide aanu?", "mal_Latn")           # Mixed WhatsApp text
]

texts = [t[0] for t in test_cases]
expected = [t[1] for t in test_cases]

results = model.batch_predict(texts, batch_size=4)

print("\nResults:")
for i, res in enumerate(results):
    text, label, score, model_used = res
    print(f"Text: {text} \nPredicted: {label} \nScore: {score:.4f} \nRouter: {model_used}\n")
    assert label == expected[i], f"Test failed for '{text}'! Expected {expected[i]} but got {label}"

# Force BERT test to ensure fallback path works
print("Forcing direct BERT test to exercise fallback path...")
direct_bert_result = model.IndicBERT_roman_inference([(0, "Njan innu busy aanu")], {}, 1)

# Extract the first result from the dictionary values
first_result = list(direct_bert_result.values())[0]
assert first_result[1] == "mal_Latn", "Direct BERT test failed!"

print("\nAll smoke tests passed! Full inference entry point verified.")


--- Running Smoke Tests ---

Results:
Text: ഞാൻ ഇന്ന് ബിസി ആണ് 
Predicted: mal_Mlym 
Score: 1.0000 
Router: IndicLID-FTN

Text: Njan innu busy aanu 
Predicted: mal_Latn 
Score: 1.0000 
Router: IndicLID-FTR

Text: I am busy today 
Predicted: eng_Latn 
Score: 0.9191 
Router: IndicLID-FTR

Text: phone evide aanu? 
Predicted: mal_Latn 
Score: 0.9735 
Router: IndicLID-FTR

Forcing direct BERT test to exercise fallback path...

All smoke tests passed! Full inference entry point verified.
